# core

> Fill in a module description here

In [ ]:
# | default_exp audio_video

In [ ]:
# | hide
from nbdev.showdoc import *  # type: ignore

## Imports

In [4]:
# | export
from typing import List, Tuple, cast, Union, Sequence
from pathlib import Path
from seewav import visualize
import cairo
import subprocess as sp
from multiprocess import Pool
import tqdm
import os
from moviepy.editor import (
    VideoFileClip,
    AudioFileClip,
    CompositeVideoClip,
    TextClip,
    concatenate_videoclips,
)
from pydantic import BaseModel, TypeAdapter
import assemblyai as aai
import requests
import os
import numpy as np
from dotenv import load_dotenv
from product_horse.db import (
    UtteranceSegment,
    AbstractDatabase,
    Word,
    UnvalidatedClip,
    UnvalidatedVideo,
    Video,
    RenderStatus,
    Clip,
    VideoType,
    User,
)
from product_horse.filesystems import AbstractFileSystem, FilePathType
from uuid import UUID
from rich.console import Console

load_dotenv()

True

## AUDIO

In [ ]:
# | export
class AssemblyAiWord(BaseModel):
    text: str
    start: int
    end: int
    confidence: float
    speaker: str


word_validator = TypeAdapter(List[AssemblyAiWord])


class AssemblyAiUtterance(BaseModel):
    confidence: float
    end: int
    speaker: str
    start: int
    text: str
    words: List[AssemblyAiWord]


class AssemblyAiTranscript(BaseModel):
    text: str
    utterances: List[AssemblyAiUtterance]


class AudioEditor:
    api_key: str | None = None

    def __init__(
        self,
        api_key: str | None = os.getenv("ASSEMBLY_API_KEY")
        if os.getenv("ASSEMBLY_API_KEY")
        else None,
    ):
        if not api_key:
            raise ValueError("ASSEMBLYAI_API_KEY environment variable not found")
        self.api_key = api_key

    def upload_for_transcript(self, file_path: str) -> str:
        """Uploads a file for transcription and returns the file URL."""
        if not file_path.startswith("https://"):
            with open(file_path, "rb") as file:
                audio_file = file.read()
        else:
            audio_file = requests.get(file_path).content
        if not self.api_key:
            raise ValueError("ASSEMBLYAI_API_KEY environment variable not found")
        response = requests.post(
            "https://api.assemblyai.com/v2/upload",
            headers={
                "Authorization": self.api_key,
                "Content-Type": "application/octet-stream",
            },
            data=audio_file,
        )
        if response.status_code != 200:
            raise Exception(f"Failed to upload file: {response.text}")
        file_url = response.json()["upload_url"]
        return file_url

    async def transcribe(self, file_url: str) -> AssemblyAiTranscript:
        """Transcribes a file and returns the transcript."""
        aai.settings.api_key = self.api_key
        config = aai.TranscriptionConfig(speaker_labels=True)
        transcript = aai.Transcriber().transcribe(file_url, config)
        text_well_formed: bool = (
            transcript.utterances
            and transcript.text
            and len(transcript.text) > 0
            and len(transcript.utterances) > 0
        )  # type: ignore
        if not text_well_formed:
            raise Exception(f"No text or utterances found in audio file: {file_url}")
        if transcript.status != "error" and text_well_formed:
            # Convert each utterance and its words to the correct Pydantic models
            transcript_text = "\n".join(
                [
                    f"Speaker {utterance.speaker}: {utterance.text}"
                    for utterance in transcript.utterances
                ]
            )
            utterances = [
                AssemblyAiUtterance(
                    confidence=u.confidence,
                    end=u.end,
                    speaker=u.speaker or "speaker not detected",
                    start=u.start,
                    text=u.text,
                    words=word_validator.validate_python(
                        u.words, from_attributes=True
                    ),  # Convert words for each utterance
                )
                for u in transcript.utterances
            ]
            return AssemblyAiTranscript(text=transcript_text, utterances=utterances)
        else:
            raise Exception(f"{transcript.error} - {transcript.status}")

    def make_webvtt(self, utterances: List[AssemblyAiUtterance]):
        # https://developer.mozilla.org/en-US/docs/Web/API/WebVTT_API
        # add later
        pass

In [ ]:
FILE_URL = "https://github.com/AssemblyAI-Examples/audio-examples/raw/main/20230607_me_canadian_wildfires.mp3"

In [ ]:
def test_upload_for_transcript():
    api_key = os.getenv("ASSEMBLYAI_API_KEY")
    if not api_key:
        raise ValueError("ASSEMBLYAI_API_KEY environment variable not found")
    audio_editor = AudioEditor(api_key=api_key)
    for url in [FILE_URL, "../bleat.m4a"]:
        file_url = audio_editor.upload_for_transcript(url)
        assert file_url is not None, "File URL is None"
        assert isinstance(file_url, str), "File URL is not a string"


test_upload_for_transcript()

In [ ]:
EXPECTED_TEXT = """Speaker A: Smoke from hundreds of wildfires in Canada is triggering air quality alerts throughout the US. Skylines from Maine to Maryland to Minnesota are gray and smoggy. And in some places, the air quality warnings include the warning to stay inside. We wanted to better understand what's happening here and why, so we called Peter DiCarlo, an associate professor in the department of Environmental Health and Engineering at Johns Hopkins University. Good morning. Professor Good morning. So what is it about the conditions right now that have caused this round of wildfires to affect so many people so far away?
Speaker B: Well, there's a couple of things. The season has been pretty dry already, and then the fact that we're getting hit in the US is because there's a couple of weather systems that are essentially channeling the smoke from those Canadian wildfires through Pennsylvania into the mid Atlantic and the northeast and kind of just dropping the smoke there.
Speaker A: So what is it in this haze that makes it harmful? And I'm assuming it is.
Speaker B: Is it is. The levels outside right now in Baltimore are considered unhealthy. And most of that is due to what's called particulate matter, which are tiny particles, microscopic, smaller than the width of your hair, that can get into your lungs and impact your respiratory system, your cardiovascular system, and even your neurological, your brain.
Speaker A: What makes this particularly harmful? Is it the volume of particulate? Is it something in particular? What is it exactly? Can you just drill down on that a little bit more? Yeah.
Speaker B: So the concentration of particulate matter I was looking at, some of the monitors that we have was reaching levels of what are, in science speak, 150 micrograms per meter cubed, which is more than ten times what the annual average should be and about four times higher than what you're supposed to have on a 24 hours average. And so the concentrations of these particles in the air are just much, much higher than we typically see. And exposure to those high levels can lead to a host of health problems.
Speaker A: And who is most vulnerable? I noticed that in New York City, for example, they're canceling outdoor activities. And so here it is in the early days of summer, and they have to keep all the kids inside. So who tends to be vulnerable in a situation like this?
Speaker B: It's the youngest. So children, obviously, whose bodies are still developing, the elderly who know their bodies are more in decline and they're more susceptible to the health impacts of breathing, the poor air quality. And then people who have preexisting health conditions, people with respiratory conditions or heart conditions, can be triggered by high levels of air pollution.
Speaker A: Could this get worse?
Speaker B: That's a good, in some areas it's much worse than others, and it just depends on kind of where the smoke is concentrated. I think New York has some of the higher concentrations right now, but that's going to change as that air moves away from the New York area. But over the course of the next few days, we will see different areas being hit at different times with the highest concentrations. I was going to ask you more fires start burning. I don't expect the concentrations to go up too much higher.
Speaker A: I was going to ask you, and you started to answer this, but how much longer could this last? Or, forgive me if I'm asking you to speculate, but what do you think?
Speaker B: Well, I think the fires are going to burn for a little bit longer, but the key for us in the US is the weather system changing. And so right now it's kind of the weather systems that are pulling that air into our mid atlantic and northeast region. As those weather systems change and shift, we'll see that smoke going elsewhere and not impact us in this region as much. And so I think that's going to be the defining factor. And I think the next couple of days we're going to see a shift in that weather pattern and start to push the smoke away from where we are.
Speaker A: And finally, with the impacts of climate change, we are seeing more wildfires. Will we be seeing more of these kinds of wide ranging air quality consequences or circumstances?
Speaker B: I mean, that is one of the predictions for climate change. Looking into the future, the fire season is starting earlier and lasting longer, and we're seeing more frequent fires. So, yeah, this is probably something that we'll be seeing more frequently. This tends to be much more of an issue in the western Us. So the eastern us getting hit right now is a little bit new. But, yeah, I think with climate change moving forward, this is something that is going to happen more frequently.
Speaker A: That's Peter DiCarlo, associate professor in the department of Environmental Health and engineering at Johns Hopkins University. Thanks so much for joining us and sharing this expertise with us.
Speaker B: Thank you for having me."""
EXPECTED_UTTERANCES_COUNT = 16


async def test_transcribe():
    api_key = os.getenv("ASSEMBLYAI_API_KEY")
    if not api_key:
        raise ValueError("ASSEMBLYAI_API_KEY environment variable not found")
    audio_editor = AudioEditor(api_key=api_key)
    transcript = await audio_editor.transcribe(FILE_URL)
    print(transcript)
    print(len(transcript.utterances))
    assert "New York" in transcript.text, "Transcript text does not match expected text"
    assert (
        len(transcript.utterances) >= EXPECTED_UTTERANCES_COUNT
    ), "Number of utterances does not match expected count"
    # check words exist
    assert all(
        utterance.words for utterance in transcript.utterances
    ), "Some utterances do not have words"


await test_transcribe()

## VIDEO

In [5]:
import uuid
from IPython.core.display import Video as NbVideo
from IPython.display import display

console = Console()

uuid.uuid4()

UUID('c84c6e90-aeb4-45e7-a080-532af161c1dd')

In [10]:
# | export
def read_audio(audio_path, seek=None, duration=None):
    command = ['ffmpeg', '-y', '-loglevel', 'panic']
    if seek is not None:
        command += ['-ss', str(seek)]
    command += ['-i', str(audio_path)]
    if duration is not None:
        command += ['-t', str(duration)]
    command += ['-f', 'f32le', '-acodec', 'pcm_f32le', '-ar', '44100', '-ac', '1', '-']
    
    proc = sp.Popen(command, stdout=sp.PIPE, stderr=sp.DEVNULL)
    wav = np.frombuffer(proc.stdout.read(), dtype=np.float32)
    return wav.reshape(-1, 1).T, 44100

def envelope(wav, window, stride):
    wav = np.pad(wav, window // 2)
    cumsum = np.cumsum(np.maximum(wav, 0))
    return 1.9 * (1 / (1 + np.exp(-2.5 * (cumsum[window:] - cumsum[:-window]) / window)) - 0.5)[::stride]

In [35]:
def fast_visualize(audio, out, seek=None, duration=None, rate=60, bars=50, 
              fg_color=(0.2, 0.5, 0.8), bg_color=(1, 1, 1), size=(640, 480)):

    audio_path = Path(audio)
    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    out_path = Path(out)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    wav, sr = read_audio(audio_path, seek=seek, duration=duration)
    wav = wav.mean(0)
    wav /= wav.std()

    window = int(sr * 0.4 / bars)
    stride = window // 5
    env = envelope(wav, window, stride)
    env = np.pad(env, (bars // 2, 2 * bars))

    frames = int(rate * (len(wav) / sr))
    smooth = np.hanning(bars)

    # Pre-calculate all frame envelopes
    all_envs = np.zeros((frames, bars))
    for idx in range(frames):
        pos = (idx / rate * sr) / stride / bars
        off = int(pos)
        loc = pos - off
        env1 = env[off * bars:(off + 1) * bars]
        env2 = env[(off + 1) * bars:(off + 2) * bars]
        w = 1 / (1 + np.exp(-4 * (loc - 0.5)))
        denv = (1 - w) * env1 + w * env2
        all_envs[idx] = denv * smooth

    print("Generating frames...")

    frame_buffers = []
    with Pool() as pool:
        args = [(idx, all_envs[idx], fg_color, bg_color, size) for idx in range(frames)]
        frame_buffers = list(pool.imap(process_frame_to_memory, args))
    print(f"Generated {len(frame_buffers)} frames in memory")

    print("Encoding animation...")
    ffmpeg_cmd = [
        "ffmpeg", "-y", "-loglevel", "warning",
        "-f", "image2pipe", "-r", str(rate), "-i", "-",
        "-i", str(audio_path.resolve()),
        "-c:v", "libx264",
        "-crf", "23", "-preset", "ultrafast",
        "-c:a", "aac", "-b:a", "128k",
        "-hwaccel", "auto",
        "-threads", "0",
        "-shortest",
        str(out_path)
    ]

    try:
        process = sp.Popen(ffmpeg_cmd, stdin=sp.PIPE, stdout=sp.PIPE, stderr=sp.PIPE) #this is faster than subprocess
        for frame_buffer in frame_buffers:
            process.stdin.write(frame_buffer.getvalue())
        process.stdin.close()
        stdout, stderr = process.communicate()
        if process.returncode != 0:
            print("FFmpeg error:", stderr.decode())
            raise RuntimeError(f"FFmpeg command failed with return code {process.returncode}")
        print("FFmpeg output:", stdout.decode())
    except ValueError:
        # This can occur if FFmpeg finishes and closes the pipe before we're done writing
        # It's usually not a problem if the video is created successfully
        print("Warning: flush of closed file occurred, but this may not affect the output")
    except BrokenPipeError:
        print("Warning: Broken pipe error occurred, but this may not affect the output")
    except Exception as e:
        print("Error during FFmpeg encoding:", str(e), type(e))
        raise

    if not out_path.exists():
        raise RuntimeError(f"Expected output file {out_path} was not created")
    else:
        print(f"Video saved successfully to {out_path}")

@lru_cache(maxsize=1000)
def generate_frame(env_tuple, fg_color, bg_color, size):
    env = np.array(env_tuple)

    global last_env, last_buffer
    if np.max(np.abs(env)) < 0.1:  # Adjust this threshold as needed
        return generate_empty_waveform(fg_color, bg_color, size)
    if last_env is not None and np.allclose(env, last_env, atol=threshold):
        return last_buffer
    last_env = env.copy()
    precision = 0.2
    surface = cairo.ImageSurface(cairo.FORMAT_ARGB32, *size)
    ctx = cairo.Context(surface)
    ctx.scale(*size)

    ctx.set_source_rgb(*bg_color)
    ctx.paint()

    T = len(env)
    width = 1. / (T * 1.2)
    pad = 0.1 * width
    delta = 2 * pad + width

    ctx.set_line_width(width)
    for step, height in enumerate(env):
        ctx.set_source_rgb(*fg_color)
        x = pad + step * delta
        ctx.move_to(x, 0.5 - 0.5 * height)
        ctx.line_to(x, 0.5)
        ctx.stroke()
        ctx.set_source_rgba(*fg_color, 0.8)
        ctx.move_to(x, 0.5)
        ctx.line_to(x, 0.5 + 0.45 * height)
        ctx.stroke()

    buffer = io.BytesIO()
    surface.write_to_png(buffer)
    buffer.seek(0)
    last_buffer = buffer
    return buffer

@lru_cache(maxsize=1)
def generate_empty_waveform(fg_color, bg_color, size):
    surface = cairo.ImageSurface(cairo.FORMAT_ARGB32, *size)
    ctx = cairo.Context(surface)
    ctx.scale(*size)

    # Fill background
    ctx.set_source_rgb(*bg_color)
    ctx.paint()

    # Draw a flat line in the middle
    ctx.set_source_rgb(*fg_color)
    ctx.set_line_width(0.01)
    ctx.move_to(0, 0.5)
    ctx.line_to(1, 0.5)
    ctx.stroke()

    buffer = io.BytesIO()
    surface.write_to_png(buffer)
    buffer.seek(0)
    return buffer

def process_frame_to_memory(args):
    idx, env, fg_color, bg_color, size = args
    return generate_frame(tuple(env), fg_color, bg_color, size)

In [6]:
# | export
def make_mp4_animation(
    audio_path: str,
    output_path: str,
    tmp_directory: str,
    seek: int = 0,  # start at the beginning
    duration: int = 10,  # duration in seconds
    rate: int = 24,  # frames per second
    bars: int = 30,  # number of bars in the visualization
    speed: int = 5,  # speed of transitions
    time: float = 0.5,  # amount of audio shown at once on a frame
    oversample: int = 5,  # frequency of changes
    fg_color: Tuple[float, float, float] = (0.2, 0.5, 0.8),  # foreground color
    bg_color: Tuple[float, float, float] = (1, 1, 1),  # background color
    size: Tuple[int, int] = (640, 480),  # size of the output video
) -> None:
    """
    Generate the visualisation for the `audio` file, using a `tmp` folder and saving the final
    video in `out`.
    `seek` and `durations` gives the extract location if any.
    `rate` is the framerate of the output video.

    `bars` is the number of bars in the animation.
    `speed` is the base speed of transition. Depending on volume, actual speed will vary
        between 0.5 and 2 times it.
    `time` amount of audio shown at once on a frame.
    `oversample` higher values will lead to more frequent changes.
    `fg_color` is the rgb color to use for the foreground.
    `bg_color` is the rgb color to use for the background.
    `size` is the `(width, height)` in pixels to generate.
    """
    fast_visualize(
        audio=Path(audio_path),
        out=Path(output_path),
        seek=seek,
        # tmp=Path(tmp_directory),
        duration=duration,
        rate=rate,
        bars=bars,
        fg_color=fg_color,
        bg_color=bg_color,
        size=size,
    )

In [36]:
# Test the make_mp4_animation function using nbdev
def test_make_mp4_animation():
    from pathlib import Path
    import os

    audio_path = "../temp-videos/wildfire.mp3"
    output_path = "test_output_wildfires.mp4"
    tmp_directory = "test_tmp"

    # Create temporary directories and files for testing
    os.makedirs(tmp_directory, exist_ok=True)
    Path(audio_path).touch()

    try:
        make_mp4_animation(
            audio_path=audio_path,
            output_path=output_path,
            tmp_directory=tmp_directory,
            seek=0,
            duration=30,
            rate=24,
            bars=30,
            speed=5,
            time=0.5,
            oversample=5,
            fg_color=(0.2, 0.5, 0.8),
            bg_color=(1, 1, 1),
            size=(640, 480),
        )
        assert Path(output_path).exists(), "Output file was not created."
    finally:
        # Clean up temporary files
        if Path(tmp_directory).exists():
            for file in os.listdir(tmp_directory):
                os.remove(os.path.join(tmp_directory, file))
            os.rmdir(tmp_directory)


try:
    test_make_mp4_animation()
    console.print("PASS!!!", style="green b")
    display(NbVideo("test_output_wildfires.mp4", embed=True))
except Exception as e:
    console.print(e, style="red")

Generating frames...
Generated 720 frames in memory
Encoding animation...
Video saved successfully to test_output_wildfires.mp4


PASS!!!

In [ ]:
# | export
from collections import OrderedDict

MoviePyClipType = Union[VideoFileClip, AudioFileClip, CompositeVideoClip]


def _group_utterances_by_transcript_id(
    utterances: Sequence[UtteranceSegment],
) -> Tuple[OrderedDict[str, List[UtteranceSegment]], List[str]]:
    """Groups utterances by transcript_id using defaultdict.
    returns:
    1. an ordered dictionary where each key is a transcript_id
        and the value is a list of utterances associated with that ID.
    2. a list of transcript_ids"""
    transcript_utterances_by_id: OrderedDict[str, List[UtteranceSegment]] = (
        OrderedDict()
    )
    transcript_ids: List[str] = []
    for utterance in utterances:
        transcript_id = str(utterance.transcription_id)
        if transcript_id not in transcript_utterances_by_id:
            transcript_utterances_by_id[transcript_id] = []
            transcript_ids.append(transcript_id)
        transcript_utterances_by_id[transcript_id].append(utterance)
    return transcript_utterances_by_id, transcript_ids


def _get_video_file_properties(file_path: str):
    clip = VideoFileClip(file_path)
    frame_rate: int = cast(int, clip.fps)
    resolution: Tuple[int, int] = cast(Tuple[int, int], clip.size)
    duration: float = clip.duration
    aspect_ratio: float = clip.aspect_ratio
    clip.close()
    return frame_rate, resolution, duration, aspect_ratio


def _get_audio_file_properties(file_path: str):
    clip = AudioFileClip(file_path)
    clip.close()
    return clip.fps, None, clip.duration

In [ ]:
def test_group_utterances_by_transcript_id():
    uuid1 = uuid.uuid4()
    uuid2 = uuid.uuid4()

    utterances = [
        UtteranceSegment(
            transcription_id=uuid1,
            custom_start=0.0,
            custom_end=1.0,
            confidence=0,
            start=0,
            end=1,
            text="test",
            words=[],
            transcription=[],
            id=uuid.uuid4(),
        ),
        UtteranceSegment(
            transcription_id=uuid1,
            custom_start=1.0,
            custom_end=2.0,
            confidence=0,
            start=1,
            end=2,
            text="test",
            words=[],
            transcription=[],
            id=uuid.uuid4(),
        ),
        UtteranceSegment(
            transcription_id=uuid2,
            custom_start=0.0,
            custom_end=1.0,
            confidence=0,
            start=0,
            end=1,
            text="test",
            words=[],
            transcription=[],
            id=uuid.uuid4(),
        ),
    ]
    grouped, ids = _group_utterances_by_transcript_id(utterances)
    console.print("grouped", grouped)
    console.print("ids", ids)
    print("len(grouped)", len(grouped))
    print("len(ids)", len(ids))
    assert len(grouped) == 2
    assert len(ids) == 2
    assert ids == [str(uuid1), str(uuid2)]
    assert len(grouped[str(uuid1)]) == 2
    assert len(grouped[str(uuid2)]) == 1


def test_get_video_file_properties():
    file_path = "test_output_wildfires.mp4"
    frame_rate, resolution, duration, aspect_ratio = _get_video_file_properties(
        file_path
    )
    print("frame_rate", frame_rate)
    print("resolution", resolution)
    print("duration", duration)
    print("aspect_ratio", aspect_ratio)
    assert frame_rate > 0
    assert resolution == [1280, 720]
    assert duration > 0
    assert aspect_ratio > 0


def test_get_audio_file_properties():
    file_path = "../temp-videos/wildfire.mp3"
    frame_rate, _, duration = _get_audio_file_properties(file_path)
    print("frame_rate", frame_rate)
    print("duration", duration)
    assert frame_rate > 0
    assert duration > 0


try:
    test_get_audio_file_properties()
    test_get_video_file_properties()
    test_group_utterances_by_transcript_id()
    console.print("PASS!!!", style="green b")
except Exception as e:
    console.print(e, style="red")

In [ ]:
# | export
def put_words_over_video_subset(
    file_path: str,
    words: Sequence[Word],
    target_resolution: Tuple[int, int],
    start: float,
    end: float = 0.0,
    position: str = "center",
) -> Tuple[CompositeVideoClip, List[Word]]:
    """Adds words to a video clip.
    - start: The starting time (in seconds) of the video segment.
    - end: The ending time (in seconds) of the video segment. If end is 0.0, the entire video from start to the end is used.
    - word_start is indexed against the start of the transcript, not necessarily the video clip. So the start of the video clip is used as an offset."""
    console = Console()
    if len(words) == 0:
        raise ValueError("No words to add to video")
    clips_with_words = []
    previous_word_end = 0.0
    gap_between_words = 0.0
    # start of this video_clip should be used as 0
    try:  # catching exceptions on VideoFileClip creation
        if end > 0:
            video_clip = VideoFileClip(
                file_path, target_resolution=target_resolution
            ).subclip(start, end)
        else:
            video_clip = VideoFileClip(file_path, target_resolution=target_resolution)
        text_positions = {
            "center": ("center", "center"),
            "bottom-third": ("center", "bottom"),
        }
    except Exception as e:
        console.print(e, style="red")
        raise e
    try:
        check_frame = video_clip.get_frame(0)
        if check_frame is None:
            raise ValueError("Video clip is has no video")
    except Exception as e:
        console.print(e, style="red")
        raise e
    text_position = text_positions.get(position, (0.5, 0.5))
    words_used: List[Word] = []
    for word in words:
        word_start = float(word.start) / 1000
        word_start_adj = word_start - start
        word_end = float(word.end) / 1000
        word_end_adj = word_end - start
        final_end = video_clip.duration
        if word_start_adj > final_end:
            console.print("word_start_adj is greater than final_end", style="red")
            final_word_clip = concatenate_videoclips(clips_with_words).set_position(
                text_position
            )
            final_clip = CompositeVideoClip([video_clip, final_word_clip])
            return final_clip, words_used  # EARLY RETURN
        if word_end_adj > final_end:
            print(
                "Word ends at a time greater than utterance.final_end. Using word as final."
            )
            word_end_adj = final_end
        word_text = word.text
        word_color = (
            "white"
            if word.speaker == "A"
            else "yellow"
            if word.speaker == "B"
            else "grey"
        )
        word_clip = TextClip(
            word_text,
            fontsize=48,
            size=(
                target_resolution[1],
                target_resolution[0] / 3,
            ),  # because of the weird resolution flipping issue
            color=word_color,
            stroke_color="black",
            font="Helvetica",
            stroke_width=2,
        )
        words_used.append(word)
        duration = word_end_adj - word_start_adj
        word_clip = word_clip.set_duration(
            duration
            + gap_between_words  # gap between words seems to be pushing a little too far ahead vs audio, revisit
        )  # setting position here doesn't seem to work at all
        clips_with_words.append(word_clip)
        gap_between_words = word_start_adj - previous_word_end
        previous_word_end = word_end_adj
    final_word_clip = concatenate_videoclips(clips_with_words).set_position(
        text_position
    )
    final_clip = CompositeVideoClip([video_clip, final_word_clip])
    return final_clip, words_used

- Maybe in the future you'll want to add some kind of ClipOperation table to track what's being done to the clip to do/undo things. As of 6/13/2024 it seems overkill

In [ ]:
def test_put_words_over_video_subset():
    file_path = "test_output_wildfires.mp4"
    words = [
        Word(start=0, end=1000, text="Hello", speaker="A", confidence=0.9),
        Word(start=1500, end=2500, text="world", speaker="B", confidence=0.9),
    ]
    target_resolution = (720, 480)
    start = 0.0
    end = 3.0
    position = "center"

    final_clip, words_used = put_words_over_video_subset(
        file_path, words, target_resolution, start, end, position
    )

    assert final_clip is not None, "Final clip should not be None"
    assert len(words_used) == 2, "Should have used 2 words"
    assert words_used[0].text == "Hello", "First word should be 'Hello'"
    assert words_used[1].text == "world", "Second word should be 'world'"
    return final_clip


try:
    video_data: CompositeVideoClip = test_put_words_over_video_subset()
    console.print("PASS!!!", style="green b")
    file_path = "test_output_wildfires_words.mp4"
    video_data.write_videofile(
        file_path,
        fps=30,
        audio_codec="aac",
        preset="ultrafast",
        codec="libx264",
        threads=os.cpu_count(),
    )
    display(NbVideo(file_path))
except Exception as e:
    console.print(e, style="red")

In [ ]:
# | export
def save_and_render_utterances_to_video(
    original_file_path: str,
    transcription_id: str,
    utterances: List[UtteranceSegment],
    duration: float,
    frame_rate: int,
    resolution: Tuple[int, int],
    metadata_to_render: str,
    video_flag: bool,
    database: AbstractDatabase,
    file_system: AbstractFileSystem,
    user: User,
    video_id: UUID | str,
) -> Clip:
    """Renders clips from utterances for a given transcript.
    If video_flag is true, the clip is a video, otherwise it is an audio clip.
    All words are pulled from the database for each utterance."""
    console = Console()
    if file_system.check_exists(original_file_path) is False:
        raise ValueError(f"File not found: {original_file_path}")
    render_buffer: List[MoviePyClipType] = []  # revisit cliptype
    user_clip_path = file_system.build_user_path(user, FilePathType.USER_ID_BASE_CLIPS)
    final_clip_start = 0.0
    final_clip_end = 0.0
    all_words_used: List[Word] = []
    with file_system.temporary_user_directory(user) as temp_directory:
        temp_audio_path_base = f"{temp_directory}/audio/"
        temp_visualization_path_base = f"{temp_directory}/visualizations/"
        file_system.create_folder(temp_audio_path_base)
        file_system.create_folder(temp_visualization_path_base)
        last_clip_number = len(utterances) - 1
        for clip_number, utterance in enumerate(utterances):
            console.print(
                f"Rendering clip {clip_number} of {last_clip_number}", style="green"
            )
            temp_audio_path = f"{temp_audio_path_base}/{clip_number}.mp3"
            temp_visualization_path = (
                f"{temp_visualization_path_base}/{clip_number}.mp4"
            )
            # this means that all utterance words will be pulled at this point
            clip_words = database.get_words_from_utterance_ids([utterance.id])
            if len(clip_words) == 0:
                raise ValueError("No words to add to video")
            if utterance.custom_start is not None:
                start = utterance.custom_start / 1000
            else:
                start: float = float(utterance.start) / 1000
            if utterance.custom_end is not None:
                end = utterance.custom_end / 1000
            else:
                end: float = float(utterance.end) / 1000
            duration = int(end - start)
            if not video_flag:
                """handle the audio file conversion to video"""
                working_clip: MoviePyClipType = AudioFileClip(
                    original_file_path
                ).subclip(start, end)  # type: ignore
                working_clip.write_audiofile(temp_audio_path)
                working_clip.close()
                # Visualize the audio segment
                if (
                    duration > 1
                ):  # if you remove this you will get failures with exit status 254 from ffmeg
                    make_mp4_animation(
                        audio_path=temp_audio_path,
                        output_path=temp_visualization_path,
                        tmp_directory=temp_directory,
                        duration=duration,
                        rate=frame_rate,
                        size=resolution,
                    )
                    word_clip, words_used = put_words_over_video_subset(
                        temp_visualization_path,
                        clip_words,
                        resolution,
                        start,
                        position="center",
                    )
                    render_buffer.append(word_clip)
                    all_words_used.extend(words_used)
                else:
                    console.print("weird short utterance", style="red")
            else:
                word_clip, words_used = put_words_over_video_subset(
                    original_file_path,
                    clip_words,
                    resolution,
                    start,
                    end,
                    position="bottom-third",
                )
                try:
                    frame = word_clip.get_frame(0)
                    if frame is None:
                        raise ValueError("Word clip has no video")
                except Exception as e:
                    console.print(e, style="red")
                    raise e
                finally:
                    render_buffer.append(word_clip)
                    all_words_used.extend(words_used)
            if clip_number == 0:
                final_clip_start = start
            if clip_number == last_clip_number:
                final_clip_end = end
        if len(render_buffer) == 0:
            raise ValueError("No clips to render")
        console.print(f"Rendered {len(render_buffer)} clips", style="green")
        final_clip = concatenate_videoclips(clips=render_buffer)
        # Do NOT close the clips here
        # for clip in render_buffer:
        #     clip.close()
        try:
            final_clip.get_frame(0)
        except Exception as e:
            # test
            console.print(e, style="red")
            raise e
    if "!" in metadata_to_render:
        pass
    else:
        metadata_banner = TextClip(
            metadata_to_render,
            fontsize=48,
            color="white",
            stroke_color="black",
            font="Helvetica",
            stroke_width=2,
        )
        metadata_banner = metadata_banner.set_position(
            (0.05, 0.05),
            relative=True,
        ).set_duration(final_clip.duration)  # type: ignore - moviepy types are crappy
        # CONDITIONALLY MUTATE FINAL_CLIP HERE
        final_clip = CompositeVideoClip([final_clip, metadata_banner])
        final_clip.fps = frame_rate
    if not file_system.check_exists(user_clip_path):
        file_system.create_folder(user_clip_path)
    with final_clip as video:
        video.write_videofile(
            user_clip_path + f"/{transcription_id}.mp4",
            fps=frame_rate,
            audio_codec="aac",
            preset="ultrafast",
            codec="libx264",
            threads=os.cpu_count(),
            logger=None,
        )
    unvalidated_clip = UnvalidatedClip(
        utterance_start=final_clip_start,
        utterance_end=final_clip_end,
        duration=final_clip_end - final_clip_start,
        fps=frame_rate,
        text=utterance.text,
        resolution_x=resolution[0],
        resolution_y=resolution[1],
        metadata_to_render=metadata_to_render,
        video_type=VideoType.video if video_flag else VideoType.audio,
        file_path=user_clip_path + f"/{transcription_id}.mp4",
    )
    clip = database.save_clip(all_words_used, unvalidated_clip, video_id)
    return clip

In [ ]:
from testcontainers.postgres import PostgresContainer
from product_horse.db import (
    SqlModelDatabase,
    UnvalidatedUtterance,
    Word,
    UnvalidatedTranscription,
    UnvalidatedFileMetadata,
)
from product_horse.filesystems import LocalFileSystem

# Start a test container for the database
container = PostgresContainer("postgres:15")
container.start()
db_url = container.get_connection_url()  # type: ignore

database = SqlModelDatabase(db_url)  # type: ignore

# Create a local file system
file_system = LocalFileSystem(
    base_path="/Users/max/Documents/product_horse/temp-videos/temp"
)

# Create a test user
user = User(name="Test User", external_id="test_user")
user = database.save_user(user)

metadata_raw = UnvalidatedFileMetadata(
    user_id=user.id,
    file_name="test_output_wildfires_words.mp4",
    file_path="test_output_wildfires_words.mp4",
)
metadata = database.save_file_metadata(metadata_raw)

transcription = UnvalidatedTranscription(
    file_id=metadata.id,
    text="Hello world",
)
word_1 = Word(
    text="Hello",
    start=0,
    end=1000,
    speaker="A",
    confidence=0.9,
)
word_2 = Word(
    text="world",
    start=1000,
    end=2000,
    speaker="A",
    confidence=0.9,
)
utterance = UnvalidatedUtterance(
    text="Hello world",
    start=0,
    speaker="A",
    end=2000,
    confidence=0.9,
    words=[word_1, word_2],
)


# Define test inputs
original_file_path = "test_output_wildfires_words.mp4"
transcription_id = "test_transcription_id"
transcription = database.save_transcription(transcription, [utterance])
duration = 2.0
frame_rate = 24
resolution = (720, 480)
metadata_to_render = "Test Metadata"
video_flag = True
video_id = "test_video_id"

# Create a video entry in the database
video = UnvalidatedVideo(title="Test Video", duration=duration, file_path=None)
saved_video = database.save_video(video)
utterance_segments = [
    UtteranceSegment(
        **utterance.model_dump(),
        custom_start=utterance.start,
        custom_end=utterance.end,
        transcription=transcription,
        words=utterance.words,
    )
    for utterance in transcription.utterances
]

print(len(utterance_segments))

result = save_and_render_utterances_to_video(
    original_file_path,
    transcription_id,
    utterance_segments,
    duration,
    frame_rate,
    resolution,
    metadata_to_render,
    video_flag,
    database,
    file_system,
    user,
    saved_video.id,
)

# Assertions
assert result is not None
assert result.video_id == saved_video.id
assert result.duration == duration, print(result.duration, duration)
assert result.fps == frame_rate, print(result.fps, frame_rate)
assert result.resolution == resolution, print(result.resolution, resolution)

print("All tests passed.")


display(NbVideo(result.file_path))

## issues:
# Clip being saved for every word?
# Video not being returned for clips
# Clip without video not being caught at save-time.

In [ ]:
# | export
def make_video_from_utterances(
    utterances: Sequence[UtteranceSegment],
    database: AbstractDatabase,
    file_system: AbstractFileSystem,
    video_title: str = "Product Horse",
    requesting_tenant: str = "producthorse",
) -> Video:
    """List of utterances to video. Utterances will be processed in the order they are recieved.
    Add custom_start and custom_end to utterance to clip it."""
    total_duration = 0.0
    default_resolution = (1280, 720)
    default_frame_rate = 30
    utterances_by_transcript_id, list_of_transcript_ids = (
        _group_utterances_by_transcript_id(utterances)
    )
    transcript_metadatas = database.get_file_metadata_from_list_of_transcript_ids(
        list_of_transcript_ids
    )
    if len(transcript_metadatas) == 0:
        raise ValueError(
            f"No metadata found, important variables listed: {list_of_transcript_ids}"
        )
    resolutions: List[Tuple[int, int]] = [
        meta.resolution for meta in transcript_metadatas
    ]  # keep for max_resolution
    frame_rates: List[int] = [
        meta.fps for meta in transcript_metadatas
    ]  # keep for max_fps
    max_resolution: Tuple[int, int] = default_resolution
    max_fps: int = default_frame_rate
    if len(resolutions) > 0:
        max_resolution = np.amax(resolutions, axis=0)
    if len(frame_rates) > 0:
        max_fps = np.max(frame_rates)
    final_video_name = file_system.sanitize_filename(f"{video_title}.mp4")
    if not file_system.check_exists(file_system.tenant_video_path):
        file_system.create_folder(file_system.tenant_video_path)
    video: Video = database.save_video(
        UnvalidatedVideo(
            title=video_title,
            duration=total_duration,
            file_path=file_system.tenant_video_path + "/" + final_video_name,
        )
    )
    if video.file_path is None:
        raise ValueError("Video file path is None")
    for meta in transcript_metadatas:
        video_flag = meta.file_path.endswith(".mp4")
        if not file_system.check_exists(meta.file_path):
            raise ValueError(f"File {meta.file_path} does not exist")
        else:
            print(f"File {meta.file_path} exists")
        if video_flag:
            frame_rate, resolution, duration, aspect_ratio = _get_video_file_properties(
                meta.file_path
            )
        else:
            _, resolution, duration = _get_audio_file_properties(meta.file_path)
            frame_rate = np.max(frame_rates) if frame_rates else default_frame_rate
        if frame_rate is None:
            raise ValueError(
                f"No frame rate or resolution found for {video_flag and 'video' or 'audio'} file {meta.file_path}"
            )
        if resolution is not None:
            resolutions.append(resolution)
            frame_rates.append(frame_rate)
        clip_utterances = utterances_by_transcript_id[str(meta.transcription.id)]
        user = database.get_user(meta.user_id)
        metadata = user.name if user is not None else None
        if duration is None or metadata is None:
            raise ValueError(
                f"Duration: {duration}, metadata: {metadata} <- one of them's None."
            )
        if resolution is None:
            resolution = default_resolution
        if user is None:
            raise ValueError("User is None")
        save_and_render_utterances_to_video(  # not putting returned clips into memory here
            meta.file_path,
            str(meta.transcription.id),
            clip_utterances,
            duration,  # type: ignore - moviepy types are crappy
            max_fps,
            max_resolution,
            metadata,
            video_flag,
            database,
            file_system,
            user,
            str(video.id),
        )
        total_duration += duration  # type: ignore - moviepy types are crappy
    # at some point the target resolution is reversed internally in moviepy for videoclips, so reverse it here to counteract that.
    target_resolution = (int(max_resolution[1]), int(max_resolution[0]))
    clips = database.get_clips_from_video_ids([video.id])
    video_clips = [VideoFileClip(clip.file_path) for clip in clips]
    final_video = concatenate_videoclips(video_clips, method="compose")
    with final_video as video_render:
        video_render.write_videofile(
            video.file_path,
            fps=max_fps,
            audio_codec="aac",
            preset="ultrafast",
            codec="libx264",
            threads=os.cpu_count(),
            logger=None,
        )
    if file_system.check_exists(video.file_path):
        print(f"Video file {video.file_path} exists, {type(video)}")
        video.duration = final_video.duration
        video.resolution_x = target_resolution[1]
        video.resolution_y = target_resolution[0]
        video.fps = max_fps
        video.render_status = RenderStatus.complete
        database.save_video(video)
    else:
        video.status = RenderStatus.failed
        database.save_video(video)
        raise ValueError(
            f"Video file {video.file_path} does not exist -- something failed"
        )
    return video

In [ ]:
def test_make_video_from_utterances():
    # Setup
    container = PostgresContainer("postgres:15")
    container.start()
    db_url = container.get_connection_url()
    database = SqlModelDatabase(db_url)
    file_system = LocalFileSystem(
        base_path="/Users/max/Documents/product_horse/temp-videos/temp"
    )

    # Create test user
    user = database.save_user(User(name="Test User", external_id="test_user"))

    # Create test audio file
    audio_path = (
        "/Users/max/Documents/product_horse/nbs/test_output_wildfires_words.mp4"
    )
    # Create test metadata and transcription
    metadata = database.save_file_metadata(
        UnvalidatedFileMetadata(
            user_id=user.id, file_name="test_audio.mp3", file_path=audio_path
        )
    )
    transcription = database.save_transcription(
        UnvalidatedTranscription(file_id=metadata.id, text="Test transcription"),
        [
            UnvalidatedUtterance(
                text="Hello world",
                start=0,
                speaker="A",
                end=2000,
                confidence=0.9,
                words=[
                    Word(text="Hello", start=0, end=1000, speaker="A", confidence=0.9),
                    Word(
                        text="world", start=1000, end=2000, speaker="A", confidence=0.9
                    ),
                ],
            )
        ],
    )

    # Create test utterance segments
    utterance_segments = [
        UtteranceSegment(
            **transcription.utterances[0].model_dump(),
            custom_start=0,
            custom_end=2000,
            transcription=transcription,
            words=transcription.utterances[0].words,
        )
    ]

    # Run function
    result = make_video_from_utterances(
        utterance_segments, database, file_system, "Test Video"
    )

    # Assertions
    assert isinstance(result, Video), f"Expected Video, got {type(result)}"
    assert result.title == "Test Video"
    assert result.duration > 0
    assert result.resolution_x > 0
    assert result.resolution_y > 0
    assert result.fps > 0
    assert result.render_status == RenderStatus.complete
    assert file_system.check_exists(result.file_path)

    # Cleanup
    container.stop()
    return result.file_path


file_path = test_make_video_from_utterances()
print("All tests passed!")
display(NbVideo(file_path, embed=True))

In [ ]:
# running list of issues
# Weird issue with resolution being off I think. didn't document when it happened
# add transcript to stream

In [ ]:
from product_horse.db import SqlModelDatabase

PG_URL = "postgresql://localhost:5432/product_horse"
db = SqlModelDatabase(database_url=PG_URL)
users = db.get_all_users()
transcripts = db.get_all_unique_transcriptions(mode="file_name")
test_transcript = transcripts[2]
test_transcript_2 = transcripts[5]
test_transcript_3 = transcripts[10]
test_transcript_4 = transcripts[11]
test_transcript_5 = transcripts[20]
test_transcripts = [
    test_transcript,
    test_transcript_2,
    test_transcript_3,
    test_transcript_4,
    test_transcript_5,
]
utterance_segments = []
for transcription in test_transcripts[0:2]:
    utterance_segments += [
        UtteranceSegment(
            **utterance.model_dump(),
            custom_start=utterance.start,
            custom_end=utterance.end,
            transcription=transcription,
            words=utterance.words,
        )
        for utterance in transcription.utterances[10:12]
    ]
print(len(utterance_segments))
total_duration = sum(
    [utterance.end - utterance.start for utterance in utterance_segments]
)
print(total_duration / 1000)

make_video_from_utterances(utterance_segments, db, file_system)

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore